# dev26 GMM 군집 소속도＋지도 LR
181과같은피처·분할. 군집은학습데이터로만추정하고클래스판정은라벨로학습.


In [1]:
import nbformat
n=nbformat.read("notebooks/181_cn7_dev26_supervised_gmm.ipynb",as_version=4)
old="\n".join(c.source for c in n.cells if c.cell_type=="code")
exec(old[:old.index("MODELS=")])
from sklearn.linear_model import LogisticRegression
MODELS=["GMM2_LR","GMM3_LR","GMM5_LR","GMM3LR_RF_mean"]
def train_heads(x,yy):
 normal=yy==0;gas=yy==1
 imp=SimpleImputer(keep_empty_features=True).fit(x[normal])
 a=imp.transform(x);sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-20,20)
 fit={"imputer":imp,"scale":sc,"gas_count":int(gas.sum())}
 if not gas.any():fit["empty_gas"]=True;return fit
 for k in [2,3,5]:
  gm=GaussianMixture(n_components=k,covariance_type="diag",reg_covar=.1,n_init=2,max_iter=150,random_state=42).fit(a)
  memberships=gm.predict_proba(a)
  lr=LogisticRegression(C=.1,class_weight="balanced",max_iter=2000,random_state=42).fit(memberships,yy)
  fit[str(k)]=(gm,lr)
 fit["RF"]=RandomForestClassifier(n_estimators=60,max_depth=4,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2).fit(a,yy)
 return fit
def scores(fit,x):
 if fit.get("empty_gas"):return {n:np.zeros(len(x)) for n in MODELS}
 a=np.clip(fit["scale"].transform(fit["imputer"].transform(x)),-20,20)
 out={}
 for k in [2,3,5]:
  gm,lr=fit[str(k)]
  out[f"GMM{k}_LR"]=lr.predict_proba(gm.predict_proba(a))[:,1]
 out["GMM3LR_RF_mean"]=(out["GMM3_LR"]+fit["RF"].predict_proba(a)[:,1])/2
 return out
runner=old[old.index("memo={};audit=[]"):old.index('lines=["# CN7 dev26')]
runner=runner.replace("cn7_181","cn7_182")
exec(runner)
lines=["# CN7 dev26: GMM 군집 소속도 + 지도 분류","",
 "181의인과dev26계산과동일한26피처/시간블록/purge10/nested5를사용한다.",
 "GMM은학습샷센서로군집표현을학습하고,그군집소속확률2/3/5개를라벨로학습한balanced LR에넣는다. GMM만으로정답을붙인것이아니다.",
 "GMM3LR＋RF 평균도비교한다.군집학습·LR·스케일·기준은해당평가구간을제외한학습만사용한다.",
 "안쪽F1/정상오탐3%예산으로선택하고평가자료로기준을튜닝하지않는다.",
 "전체학습배포기준도안쪽crossfit에서고정한다.비라벨RPM0/로그관계불능은보류한다.","",
 "## 라벨",mm.to_string(index=False),"","## 비라벨",um.to_string(index=False),"",
 "common.py가없어원본과완전히같은실험은아니다.기록단위고정·라벨/비라벨이력밀도·동일Run시간블록검증한계는181과같다."]
(ROOT/"reports/cn7_182_dev26_gmm_supervised_membership.md").write_text("\n".join(lines),encoding="utf-8")


Completed dev26 supervised GMM 1/20


Completed dev26 supervised GMM 2/20


Completed dev26 supervised GMM 3/20


Completed dev26 supervised GMM 4/20


Completed dev26 supervised GMM 5/20


...

Completed dev26 supervised GMM 18/20


Completed dev26 supervised GMM 19/20


Completed dev26 supervised GMM 20/20



LABELED
                  model   N  gas  TP  FP  FN  TN  F1  precision  recall       AP
               GMM2_LR 605   13   0   2  13 590   0        0.0     0.0 0.183949
               GMM3_LR 605   13   0   1  13 591   0        0.0     0.0 0.199901
               GMM5_LR 605   13   0   0  13 592   0        0.0     0.0 0.097105
        GMM3LR_RF_mean 605   13   0   7  13 585   0        0.0     0.0 0.018696
primary_inner_selected 605   13   0   0  13 592   0        0.0     0.0 0.097363

UNLABELED
          model  selected_for_deployment  threshold  eligible_unknown  gas_candidates  alarm_percent
       GMM2_LR                        0   0.501484              9585            1584      16.525822
       GMM3_LR                        1   0.644757              9585            1570      16.379760
       GMM5_LR                        0   0.684101              9585               0       0.000000
GMM3LR_RF_mean                        0   0.503756              9585             423       4.41314

1354